
# Complete Step-by-Step Guide to Build a Modern RAG Application

This notebook teaches you how to build a production-ready Retrieval-Augmented Generation (RAG) application using:

- LangChain
- Mistral AI
- Weaviate Vector Database
- HuggingFace Embeddings

---

# What You Will Learn

By the end of this notebook you will know:

1. What RAG is
2. How vector databases work
3. How embeddings work
4. How to load PDF files
5. How to split documents into chunks
6. How to create embeddings
7. How to store embeddings in Weaviate
8. How to retrieve relevant information
9. How to connect an LLM
10. How to ask questions over your own documents

---

# RAG Architecture

User Question  
↓  
Retriever searches vector DB  
↓  
Relevant chunks returned  
↓  
LLM receives context + question  
↓  
LLM generates accurate answer

---

# Prerequisites

Before running this notebook:

## Step 1: Install Python

Download Python 3.10+ from:
https://www.python.org/downloads/

---

## Step 2: Install Jupyter

Open terminal or command prompt:

```bash
pip install notebook
```

Start Jupyter:

```bash
jupyter notebook
```

---

## Step 3: Create Accounts

You need:

1. Mistral AI account
2. Weaviate Cloud account

---

## Step 4: Get API Keys

### Mistral API Key
https://console.mistral.ai/

### Weaviate Cloud
https://console.weaviate.cloud/

Create a cluster and copy:

- Cluster URL
- API Key

---

## Step 5: Prepare PDF Files

Place your PDF document in the same folder as this notebook.

Example:

```bash
project/
│
├── notebook.ipynb
├── RAG.pdf
```

---

# Recommended Folder Structure

```bash
RAG_Project/
│
├── data/
│   └── RAG.pdf
│
├── notebooks/
│   └── rag.ipynb
│
├── requirements.txt
│
└── .env
```

---

# Important Concepts

## Embeddings
Embeddings convert text into numerical vectors.

Example:

"AI is amazing" → [0.123, 0.456, 0.789...]

---

## Vector Database
A vector database stores embeddings for similarity search.

We use:
- Weaviate

---

## Chunking
Large documents are split into smaller chunks before embedding.

This improves:
- Retrieval quality
- Context relevance
- LLM accuracy

---

# Execution Order

Run notebook cells in this order:

1. Install dependencies
2. Configure environment
3. Connect Weaviate
4. Load embeddings model
5. Load PDF
6. Split documents
7. Store vectors
8. Create retriever
9. Initialize Mistral
10. Build RAG chain
11. Ask questions

---

# Common Errors and Solutions

## Error: API Key Invalid
Check:
- Spaces in API key
- Wrong environment variable

---

## Error: Weaviate Connection Failed
Check:
- Cluster URL
- Internet connection
- API key permissions

---

## Error: No Module Named
Install packages again:

```bash
pip install -U package_name
```

---

# Best Practices

1. Use chunk overlap
2. Use smaller chunk sizes for precision
3. Store metadata
4. Add source citations
5. Use temperature=0 for factual QA
6. Use reranking for better retrieval
7. Monitor token usage

---

# Production Improvements

You can later add:

- FastAPI backend
- Streamlit UI
- Chat memory
- Hybrid search
- Reranking
- Authentication
- Logging
- LangSmith tracing
- Docker deployment
- Kubernetes scaling

---

# Let's Build the RAG System 🚀


# Modern RAG Application using LangChain, Mistral AI, and Weaviate

This updated notebook includes:

- Latest LangChain package structure
- Modern Weaviate client usage
- Mistral AI integration
- Production-ready RAG pipeline
- Environment variable support
- Better chunking strategy
- Retrieval QA chain
- Source citations
- Cleaner dependency management
- Compatible with latest LangChain ecosystem


In [ ]:
# Install latest dependencies

!pip install -qU \
    langchain \
    langchain-community \
    langchain-core \
    langchain-text-splitters \
    langchain-mistralai \
    langchain-weaviate \
    weaviate-client \
    pypdf \
    sentence-transformers \
    python-dotenv


In [ ]:
# Environment setup

import os
from getpass import getpass

# Add your keys here
os.environ["MISTRAL_API_KEY"] = getpass("Enter Mistral API Key: ")
os.environ["WEAVIATE_URL"] = input("Enter Weaviate URL: ")
os.environ["WEAVIATE_API_KEY"] = getpass("Enter Weaviate API Key: ")


## Connect to Weaviate

In [ ]:
import weaviate
from weaviate.auth import AuthApiKey

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=AuthApiKey(os.environ["WEAVIATE_API_KEY"])
)

print("Connected to Weaviate:", client.is_ready())


## Load Embedding Model

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


## Load PDF Documents

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "RAG.pdf"   # Replace with your PDF path

loader = PyPDFLoader(pdf_path)

documents = loader.load()

print(f"Loaded {len(documents)} pages")


## Split Documents into Chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""]
)

docs = text_splitter.split_documents(documents)

print(f"Created {len(docs)} chunks")


## Store Embeddings in Weaviate

In [ ]:
from langchain_weaviate.vectorstores import WeaviateVectorStore

vectorstore = WeaviateVectorStore.from_documents(
    documents=docs,
    embedding=embedding_model,
    client=client,
    index_name="LangChainRAGDemo",
    text_key="text"
)

print("Documents indexed successfully")


## Create Retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

retriever


## Initialize Mistral LLM

In [ ]:
from langchain_mistralai import ChatMistralAI

llm = ChatMistralAI(
    model="mistral-large-latest",
    temperature=0
)

print("Mistral model initialized")


## Build RAG Chain

In [ ]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
    chain_type="stuff"
)

print("RAG chain ready")


## Ask Questions

In [ ]:
query = "What is Retrieval-Augmented Generation?"

response = qa_chain.invoke({"query": query})

print("\nAnswer:\n")
print(response["result"])


## Display Source Documents

In [ ]:
print("\nSource Chunks:\n")

for i, doc in enumerate(response["source_documents"]):
    print(f"Source {i+1}:")
    print(doc.page_content[:500])
    print("-" * 80)


## Advanced Improvements

In [ ]:
# Optional enhancements:
#
# 1. Hybrid Search
# 2. Reranking
# 3. Metadata filtering
# 4. Multi-query retrieval
# 5. Conversational memory
# 6. Streaming responses
# 7. FastAPI or Streamlit deployment
# 8. Observability with LangSmith
# 9. Guardrails and moderation
# 10. Async processing


## Close Weaviate Connection

In [ ]:
client.close()
print("Connection closed")
